# 4장 2강: 교차검증 전략과 데이터 누수 방지

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 불균형 분류 문제에서 적절한 교차검증 방법을 선택하고, 데이터 누수를 방지하는 방법을 연습합니다.
이번 실습은 거래 간 시간 의존성을 고려하지 않는 분류 연습으로 StratifiedKFold를 사용하며, 실제 미래 거래 예측을 평가할 때는 시간 순서를 보존해 분할한다.

- KFold 교차검증의 기본 원리를 확인합니다.
- 불균형 분류에서 StratifiedKFold가 필요한 이유를 확인합니다.
- KFold / StratifiedKFold / TimeSeriesSplit의 선택 기준을 구분합니다.
- 잘못된 전처리 순서가 데이터 누수를 만들 수 있음을 이해합니다.
- `Pipeline`을 이용해 전처리와 모델 학습을 교차검증 안에서 안전하게 연결합니다.
- SMOTE를 사용할 때 `imbalanced-learn Pipeline`을 이용해 Train Fold에만 적용되도록 구성합니다.

> 이 실습에서는 4장 2강 교안에 나온 교차검증, 데이터 누수, Pipeline 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습은 불균형 분류 문제이므로 `Class` 비율을 유지하는 교차검증 전략이 중요합니다.

## 실습 준비

1. 데이터를 불러옵니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 입력 특성 X로 사용합니다.
4. `Class`를 정답 y로 사용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

data = pd.read_csv("creditcard.csv")

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(y.value_counts().to_frame("count"))

print("\n[Class 비율]")
display(y.value_counts(normalize=True).to_frame("ratio"))

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727


---

# 필수 1. KFold와 StratifiedKFold 비교

## 배경

KFold는 데이터를 여러 Fold로 나누어 반복 평가합니다.

하지만 Credit Card Fraud처럼 클래스 비율이 크게 불균형한 데이터에서는 Fold마다 사기 거래 비율이 달라질 수 있습니다.

StratifiedKFold는 각 Fold에서 원본 데이터와 비슷한 클래스 비율을 유지하도록 분할합니다.

## 문제 1. KFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `KFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Fold의 Validation 데이터에서 `Class=1` 비율을 계산합니다.
3. Fold별 사기 거래 비율을 출력합니다.

### 확인 포인트

- KFold는 클래스 비율 자체를 유지하도록 설계된 방법이 아님을 이해했는가?
- Fold별 소수 클래스 비율이 조금씩 달라질 수 있음을 확인했는가?

In [2]:
# TODO
# KFold의 각 Validation Fold에서 Class=1 비율을 확인하세요.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

kfold_ratios = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    y_val = y.iloc[val_idx]

    fraud_count = (y_val == 1).sum()
    fraud_ratio = (y_val == 1).mean()

    kfold_ratios.append({
        "Fold": f"Fold {fold}",
        "Val 전체 수": len(y_val),
        "사기 거래 수 (개)": fraud_count,
        "사기 거래 비율 (%)": f"{fraud_ratio * 100:.4f}%"
    })

kfold_df = pd.DataFrame(kfold_ratios)

print("=== KFold Validation Fold별 Class=1 비율 ===")
display(kfold_df)
print(f"전체 원본 데이터의 사기 거래 비율: {(y == 1).mean() * 100:.4f}%")

=== KFold Validation Fold별 Class=1 비율 ===


,Fold,Val 전체 수,사기 거래 수 (개),사기 거래 비율 (%)
0,Fold 1,56962,98,0.1720%
1,Fold 2,56962,93,0.1633%
2,Fold 3,56961,95,0.1668%
3,Fold 4,56961,101,0.1773%
4,Fold 5,56961,105,0.1843%


전체 원본 데이터의 사기 거래 비율: 0.1727%


### 
- 전체 28만 4천건을 다섯 묶음으로 나누면 두 검증 묶음은 56,962, 56,961건 정도로 나뉨
- 앞서 Fraud_ratio의 비율을 확인하여 각 폴드에 들어간 사기 건수는 98, 93, 95, 101, 105건, 합은 492건
###
- KFold의 묶음은 전체 크기를 비슷하게 맞추려고함, 단 클래스별 개수를 맞추도록 설계된 방법은 아님
- 이번에는 표본이 어느정도 크기때문에 유사한 크기로 사기건들이 각 폴드에 포함되지만 KFold의 각 폴드들에 데이터셋에 따라 양성이 포함되지 않을 수도 있다. 

## 문제 2. StratifiedKFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Validation Fold의 `Class=1` 비율을 계산합니다.
3. 원본 전체 데이터의 사기 거래 비율과 비교합니다.

### 확인 포인트

- 각 Fold의 클래스 비율이 원본과 비슷하게 유지되는가?
- 불균형 분류에서 StratifiedKFold가 적합한 이유를 설명할 수 있는가?

In [3]:
# TODO
# StratifiedKFold의 각 Validation Fold에서 Class=1 비율을 확인하세요.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

skfold_ratios = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    y_val = y.iloc[val_idx]

    fraud_count = (y_val == 1).sum()
    fraud_ratio = (y_val == 1).mean()
    
    skfold_ratios.append({
        "Fold": f"Fold {fold}",
        "Val 전체 수": len(y_val),
        "사기 거래 수 (개)": fraud_count,
        "사기 거래 비율 (%)": f"{fraud_ratio * 100:.4f}%"
    })


skfold_df = pd.DataFrame(skfold_ratios)

print("=== StratifiedKFold Validation Fold별 Class=1 비율 ===")
display(skfold_df)
print(f"전체 원본 데이터의 사기 거래 비율: {(y == 1).mean() * 100:.4f}%")

=== StratifiedKFold Validation Fold별 Class=1 비율 ===


,Fold,Val 전체 수,사기 거래 수 (개),사기 거래 비율 (%)
0,Fold 1,56962,99,0.1738%
1,Fold 2,56962,99,0.1738%
2,Fold 3,56961,98,0.1720%
3,Fold 4,56961,98,0.1720%
4,Fold 5,56961,98,0.1720%


전체 원본 데이터의 사기 거래 비율: 0.1727%


###
- 전체 비율을 봤을 때 0.0017, 약 0.17% 정도
- 사기 492건을 5로 나누면 98.4건, 0.4건을 쪼갤 수 없기 때문에 99건 포함하여 비교적 고르게 각 폴드에 사기 건수가 분포
###
- 층화(각 폴드의 클래스 비율을 균등하게 맞추는 것), 클래스 구성 차이를 줄이는 데 도움이 되지만
- 거래의 특성 분포까지 동일하게 맞출 수는 없기 때문에 각 폴드의 결과가 같지는 않다.

## 문제 3. 두 교차검증 방법으로 F1-score를 비교하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 사용합니다.
2. KFold로 5-Fold F1-score를 계산합니다.
3. StratifiedKFold로 5-Fold F1-score를 계산합니다.
4. 각 방법의 Fold별 점수, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 한 번의 점수가 아니라 여러 Fold의 평균과 편차를 확인했는가?
- Credit Card Fraud와 같은 불균형 분류에서는 어떤 방법이 더 안전한 기본 선택인지 설명할 수 있는가?

In [4]:
# TODO
# KFold와 StratifiedKFold의 F1-score 평균과 표준편차를 비교하세요.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

model = LogisticRegression(max_iter=1000, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_kf = cross_val_score(model, X, y, cv=kf, scoring="f1", n_jobs=-1)
scores_skf = cross_val_score(model, X, y, cv=skf, scoring="f1", n_jobs=-1)

comparison_results = pd.DataFrame(
    {
        "Fold": [f"Fold {i+1}" for i in range(5)] + ["평균 (Mean)", "표준편차 (Std)"],
        "KFold F1": list(scores_kf) + [np.mean(scores_kf), np.std(scores_kf)],
        "StratifiedKFold F1": list(scores_skf) + [np.mean(scores_skf), np.std(scores_skf)],
    }
)

print("=== KFold vs StratifiedKFold 5-Fold F1-score 비교 ===")
display(
    comparison_results.style.format(
        {"KFold F1": "{:.4f}", "StratifiedKFold F1": "{:.4f}"}
    )
)

=== KFold vs StratifiedKFold 5-Fold F1-score 비교 ===


,Fold,KFold F1,StratifiedKFold F1
0,Fold 1,0.6829,0.6909
1,Fold 2,0.7453,0.7168
2,Fold 3,0.7160,0.7073
3,Fold 4,0.7108,0.7636
4,Fold 5,0.7513,0.7614
5,평균 (Mean),0.7213,0.7280
6,표준편차 (Std),0.0249,0.0294


###
- Fold 별 값의 의미
- KFolddml 0.6829는 첫번째 묶음에서 얻은 F1-Score이며, 전체 거래의 68%를 맞혔다는 의미가 아님.
###
- 두 가지 결과를 비교하여 결론
- F1-Score가 0.72로 비교적 비슷하고, StratifiedKFold가 비교적 평균이 약간 높았다.
- 표준편차는 StratifiedKFold가 더 컸다.
- 두 가지 결과를 비교했을 때 StratifiedKFold 모델을 선택하는 근거는 극소수인 사기를 검증 묶음에 고르게 분할했기 때문이다.

### Q1. Credit Card Fraud 데이터에서는 KFold와 StratifiedKFold 중 어떤 방법을 우선 사용하는 것이 적절한가요?

**답변:**  
StratifiedKFold: 소수 클래스인 사기 거래 비율이 매우 낮은 불균형 데이터이기 때문에 더 유리할 가능성이 높다.

### Q2. 다음 상황에서 어떤 교차검증 방법을 선택할지 작성하세요.

1. 일반적인 회귀 문제
2. 불균형 분류 문제
3. 시간 순서가 중요한 시계열 문제

**답변:**  
- 1. KFold
- 2. StratifiedKFold
- 3. TimeSeriesSplit

###
- 교차검증은 무조건 하나의 방법을 고집하여 쓰는 것은 아니고 데이터의 특성에 맞게 선택한다.

---

# 필수 2. Pipeline으로 데이터 누수 방지

## 배경

전처리를 전체 데이터에 먼저 `fit()`한 뒤 교차검증하면 Validation Fold의 정보가 전처리 기준에 포함될 수 있습니다.

교안에서는 이를 방지하기 위해 **전처리와 모델을 Pipeline으로 묶는 방법**을 사용합니다.

## 문제 1. StandardScaler와 Logistic Regression을 Pipeline으로 연결하세요.

### 요구사항

1. `StandardScaler()`와 `LogisticRegression(class_weight="balanced", max_iter=1000)`을 하나의 `Pipeline`으로 만듭니다.
2. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 사용합니다.
3. `cross_val_score()`로 F1-score를 계산합니다.
4. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 각 Fold의 Train 데이터에서만 scaler가 fit되는 구조인가?
- Validation Fold에는 Train에서 학습한 기준으로 transform만 적용되는가?
- 불균형 분류이므로 StratifiedKFold를 사용했는가?

In [5]:
# TODO
# StandardScaler + LogisticRegression을 Pipeline으로 연결하고
# StratifiedKFold로 F1-score를 계산하세요.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. StandardScaler와 LogisticRegression(class_weight="balanced") 파이프라인 생성
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

# 2. StratifiedKFold 설정 (5-Fold, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. cross_val_score를 이용한 5-Fold F1-score 계산
f1_scores = cross_val_score(
    estimator=pipeline,
    X=X,
    y=y,
    cv=skf,
    scoring="f1",
    n_jobs=-1
)

# 4. Fold별 F1-score, 평균, 표준편차 출력
results_df = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(len(f1_scores))] + ["평균 (Mean)", "표준편차 (Std)"],
    "F1-score": list(f1_scores) + [np.mean(f1_scores), np.std(f1_scores)]
})

print("=== Pipeline (StandardScaler + LogisticRegression) 5-Fold F1-score ===")
display(results_df.style.format({"F1-score": "{:.4f}"}))

=== Pipeline (StandardScaler + LogisticRegression) 5-Fold F1-score ===


,Fold,F1-score
0,Fold 1,0.1282
1,Fold 2,0.1200
2,Fold 3,0.1121
3,Fold 4,0.1118
4,Fold 5,0.1175
5,평균 (Mean),0.1179
6,표준편차 (Std),0.0060


### 결과표의 의미
- 0.1118 ~ 0.1281까지 분포, F1-Score가 전반적으로 높지 않음
### 앞의 F1-Score는 0.72% 였는데 지금은 왜 0.12%인가?
- 가장 큰 차이점: 표준화와 클래스 가중치 적용 여부
- 클래스 가중치는 사기 탐지를 더 적극적으로 확인하려고 함 -> 정상을 오탐으로 예측할 확률이 증가함
###
- 전체 데이터로 sacler를 먼저 fit하면 왜 누수라고 부르는가?
- Validation은 학습 과정에서 직접 풀어보지 않은 자료의 역할을 담당한다.
- 전체 X로 먼저 fit을 하면, Validation의 입력값도 평균과 표준편차에 들어가기 때문에
- 미리 정답지를 알고 사용한 경우라고 판단하여 데이터 누수라고 본다.
- 잘못된 순서: 전체 데이터 전처리 및 Scaling -> 변환된 데이터를 cv로 분할 -> 평가

### Q3. Pipeline을 사용하면 스케일링 누수를 방지할 수 있는 이유는 무엇인가요?

**답변:**  
- 교차검증의 각 Fold마다 전처리의 fit()이 Train Fold에만 수행되기 때문

## 문제 2. SMOTE를 교차검증 안에서 안전하게 적용하세요.

### 배경

SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 Validation Fold와 매우 비슷한 합성 데이터가 Train에 포함될 수 있습니다.

교안에서는 `imbalanced-learn`의 Pipeline 안에 SMOTE를 넣어 각 Fold의 Train 부분에만 적용하도록 합니다.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. `LogisticRegression(max_iter=1000)`과 함께 `ImbPipeline`으로 연결합니다.
3. `StratifiedKFold`를 사용합니다.
4. `cross_val_score()`로 F1-score를 계산합니다.
5. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- SMOTE가 교차검증 밖에서 전체 데이터에 적용되지 않았는가?
- 각 Fold의 Train 부분에만 SMOTE가 적용되는 구조인가?

In [6]:
# TODO
# SMOTE + LogisticRegression을 imbalanced-learn Pipeline으로 연결하세요.
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. SMOTE + LogisticRegression을 imbalanced-learn Pipeline으로 구성
# (주의: scikit-learn의 기본 Pipeline은 resample 메서드를 지원하지 않으므로 ImbPipeline 사용)
pipeline_smote = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# 2. StratifiedKFold 설정 (5-Fold, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. cross_val_score로 5-Fold F1-score 계산
f1_scores_smote = cross_val_score(
    estimator=pipeline_smote,
    X=X,
    y=y,
    cv=skf,
    scoring="f1",
    n_jobs=-1
)

# 4. Fold별 F1-score, 평균, 표준편차 출력
results_smote_df = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(len(f1_scores_smote))] + ["평균 (Mean)", "표준편차 (Std)"],
    "F1-score": list(f1_scores_smote) + [np.mean(f1_scores_smote), np.std(f1_scores_smote)]
})

print("=== ImbPipeline (SMOTE + LogisticRegression) 5-Fold F1-score ===")
display(results_smote_df.style.format({"F1-score": "{:.4f}"}))

=== ImbPipeline (SMOTE + LogisticRegression) 5-Fold F1-score ===


,Fold,F1-score
0,Fold 1,0.1226
1,Fold 2,0.1107
2,Fold 3,0.1065
3,Fold 4,0.1044
4,Fold 5,0.1069
5,평균 (Mean),0.1102
6,표준편차 (Std),0.0065


### 해석
- 이번 SMOTE의 평균 F1은 0.11정도, 앞서 진행한 가중치 + 표준화보다 약 0.007 낮음
### 각 Fold에서 어떤 데이터들이 생성되는가?
- 정상건수 28만 4천 건 정도, 사기 건수 492건, 소수 클래스에 SMOTE를 적용하여 학습 표본을 크게 높임
### 전체 데이터셋이 미리 SMOTE를 적용하면 생기는 문제
- 첫 번째, Validation에 들어갈 실제 사기 표본이 합성 표본의 재료나 이웃으로 사용될 수 있다.
- -> Train에서 이미 학습한 내용들을 Validation에서 사용할 수 있기 때문에 누수가 발생할 가능성이 있다.
### 
- 두 번째, 합성한 전체 데이터를 분할하면 Validation의 사기 비율도 인위적으로 높아질 수 있음
- -> 원래 사기 건수가 매우 드물기 때문에 Precision와 F1-Score를 해석하는 조건도 달라짐.

### Q4. SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 왜 문제가 될 수 있나요?

**답변:**  
- SMOTE가 전체 데이터의 소수 클래스 샘플을 이용해 합성 데이터를 만들면, 이후 Validation Fold와 매우 비슷한 샘플이 Train에 포함될 수 있다.

---

# 과제 1. 데이터 특성에 맞는 CV와 누수 없는 Pipeline 구성

## 배경

필수 실습에서는 불균형 분류에 적합한 StratifiedKFold와 Pipeline을 직접 사용했습니다.

이번 과제에서는 같은 Credit Card Fraud 데이터에서 **누수 없는 평가 흐름을 스스로 구성하고 설명**합니다.

> 과제는 필수 실습에서 사용한 구성 요소를 다시 연결하는 수준입니다.

## 요구사항

1. `StandardScaler()`를 사용합니다.
2. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
3. 두 단계를 `Pipeline`으로 연결합니다.
4. Credit Card Fraud 데이터 특성에 맞는 교차검증 방법을 직접 선택합니다.
5. `n_splits=5`를 사용합니다.
6. `cross_val_score(..., scoring="f1")`로 성능을 평가합니다.
7. Fold별 F1-score, 평균, 표준편차를 출력합니다.
8. 왜 해당 교차검증 방법을 선택했는지 설명합니다.
9. 다음 잘못된 순서를 데이터 누수 관점에서 설명합니다.

```text
전체 X로 StandardScaler.fit_transform()
        ↓
교차검증 수행
```

10. 시계열 데이터라면 같은 방법을 그대로 사용하면 안 되는 이유와 어떤 CV를 선택해야 하는지 작성합니다.

11. `build_pipeline(model)` 함수를 작성하여 `StandardScaler`와 전달받은 모델을 연결한 Pipeline을 반환합니다. 이 함수 안에서는 데이터를 학습하거나 변환하지 않습니다.
12. `evaluate_pipeline(X, y, model, cv, scoring="f1")` 함수를 작성합니다. 11번 함수를 이용해 Pipeline을 만들고 교차검증한 뒤, Fold별 점수·평균·표준편차를 반환합니다. 모델·CV·평가지표를 함수 인자로 교체할 수 있어야 합니다.
13. 12번 함수를 실제 호출하여 요구사항 7번 결과를 출력합니다. CV에서 데이터를 섞는 경우 `shuffle=True, random_state=42`를 지정하고, 모델 등 무작위성이 있는 구성 요소도 가능한 경우 `random_state=42`로 고정합니다.
14. 함수의 재사용 방법과 재현성 설정을 설명하고 Q5~Q8에 답합니다. 별도 `.py` 모듈 제작이나 다른 모델의 추가 학습은 요구하지 않습니다.

### 확인 포인트

- 불균형 분류 특성에 맞는 CV를 선택했는가?
- Pipeline이 전처리 누수를 방지하는 구조임을 설명했는가?
- 평균뿐 아니라 Fold별 점수와 표준편차도 확인했는가?
- TimeSeriesSplit의 사용 상황을 구분할 수 있는가?
- 함수 인자로 모델·CV·평가지표를 바꿀 수 있는가?
- 함수 호출로 전처리·학습·평가가 연결되고 점수가 반환되는가?
- 재현성을 위한 난수 설정을 명시했는가?


In [8]:
# TODO
# build_pipeline(model)과 evaluate_pipeline(X, y, model, cv, scoring)를 작성하세요.
# 적절한 5-Fold CV와 모델을 전달하고 F1 점수·평균·표준편차를 출력하세요.
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# ==============================================================================
# 11. build_pipeline 함수 작성
# ==============================================================================
def build_pipeline(model):
    """
    StandardScaler와 입력받은 모델을 연결한 Pipeline 객체를 반환합니다.
    함수 내부에서는 데이터를 fit하거나 transform하지 않습니다.
    """
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    return pipeline

# ==============================================================================
# 12. evaluate_pipeline 함수 작성
# ==============================================================================
def evaluate_pipeline(X, y, model, cv, scoring="f1"):
    """
    build_pipeline을 호출해 Pipeline을 생성하고,
    전달받은 CV 전략과 scoring 지표로 교차검증을 수행하여 결과를 반환합니다.
    """
    pipeline = build_pipeline(model)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    return scores, mean_score, std_score

# ==============================================================================
# 1~5 & 13. 모델, CV 설정 및 평가 실행
# ==============================================================================
# 1~2. 모델 정의 (난수 고정)
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

# 4~5. 불균형 데이터에 적합한 StratifiedKFold 정의 (shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6 & 13. evaluate_pipeline 함수 호출 및 교차검증 수행
fold_scores, mean_f1, std_f1 = evaluate_pipeline(
    X=X,
    y=y,
    model=log_reg,
    cv=skf,
    scoring="f1"
)

# 7. Fold별 F1-score, 평균, 표준편차 출력
print("=== Credit Card Fraud: 누수 없는 Pipeline 5-Fold CV 결과 ===")
for fold_idx, score in enumerate(fold_scores, 1):
    print(f"Fold {fold_idx} F1-score : {score:.4f}")

print("-" * 40)
print(f"Mean F1-score     : {mean_f1:.4f}")
print(f"Std F1-score      : {std_f1:.4f}")

=== Credit Card Fraud: 누수 없는 Pipeline 5-Fold CV 결과 ===
Fold 1 F1-score : 0.1282
Fold 2 F1-score : 0.1200
Fold 3 F1-score : 0.1121
Fold 4 F1-score : 0.1118
Fold 5 F1-score : 0.1175
----------------------------------------
Mean F1-score     : 0.1179
Std F1-score      : 0.0060


#### Q5. 이번 Credit Card Fraud 데이터에서 선택한 CV와 그 이유를 작성하세요.

**답변:** 
- StratifiedKFold: 정상거래건이 압도적으로 많은 극단적인 클래스 불균형 데이터이기 때문에 선택했다. 일반 KFold는 랜덤 시드에 따라 사기거래건이 한 건도 없을 가능성이 있으므로 왜곡된 학습을 할 위험이 높다.

#### Q6. 전체 데이터에 StandardScaler를 먼저 fit하면 어떤 정보가 Validation Fold에서 Train 쪽으로 새어 들어갈 수 있나요?

**답변:**  
- 전체 데이터에 StandardScaler를 먼저 fit하면 validation에서 처음 분석해야할 데이터가 전처리 단계로 미리 새어 들어갈 수 있다.

### Q7. 작성한 함수를 다른 모델·CV·평가지표에 재사용하려면 어떤 인자를 바꾸면 되나요?

**답변:**
- model, cv, scoring

### Q8. 재현성을 위해 어떤 설정을 고정했으며, Pipeline 생성 함수 안에서 전체 데이터로 전처리를 미리 학습하면 안 되는 이유는 무엇인가요?

**답변:**
- random_state=42
- 전체 데이터로 미리 전처리를 학습하면 데이터 누수가 발생하여 train 데이터에 validation이나 test 데이터가 새어 들어올 수 있다.

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 교차검증을 사용하는 이유는 무엇인가요?
- 한 번의 데이터 분할 결과에만 의존하지 않고 여러번 학습하기 때문에 더 안정적인 추정치를 얻을 수 있다.

2. KFold와 StratifiedKFold의 가장 큰 차이는 무엇인가요?
- KFold는 데이터를 K로 분할
- StratifiedKFold는 각 Fold의 클래스 비율도 유지

3. 시계열 데이터에는 어떤 교차검증 방법이 적절한가요?
- TimeSeriesSplit

4. StandardScaler는 전체 데이터가 아니라 어디에서 fit되어야 하나요?
- Train 데이터에서만 fit해야함

5. SMOTE는 각 Fold의 Train / Validation 중 어디에만 적용되어야 하나요?
- Train 데이터에만 적용

6. Pipeline이 데이터 누수 방지에 도움이 되는 이유는 무엇인가요?
- 교차검증의 각 Fold마다 Train 데이터에서 전처리를 학습하고 Validation 데이터는 해당 기준을 적용하도록 과정을 묶어주기 때문